In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import os
import re
from tqdm import tqdm
from pandas.api.types import is_string_dtype
from pandas.api.types import is_numeric_dtype

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
conn = psycopg2.connect(dbname="eicu", user="username", options="-c search_path=eicu_crd")
cursor = conn.cursor()

Functions needed for data cleaning

In [ ]:
def cast_clean_float(x):
    """Remove elements interferring with float conversions
    like %"""
    try:
        x_float = float(x)    
    except ValueError as e:
        # if not just chars
        if re.match(r'.*[0-9].*', x):
            x = re.sub(r'[^0-9\.]', '', x)
            try:
                x_float = float(x) if x != '' else 0.0
            # return values like 1.3.4 as string
            except ValueError as e:
                x_float = x
        # if empty
        elif x.strip() in ['', '.']:
            x_float = float('nan')
        else:
            x_float = x
    return x_float

In [ ]:
def cast_clean_int(x):
    """Age column includes value >89 and empty values"""
    try:
        x_int = int(x)
    except ValueError as e:
        x = re.sub(r'[^0-9]', '', x)
        x_int = int(x) if x != '' else 0
    return x_int

## Raw statistics

In [ ]:
quantiles = [0, 0.1, 0.9, 1]
quantiles_2 = [0, 0.25, 0.5, 0.75, 1]

### Tables without offsets

In [ ]:
no_offset_tables = {
    #"patient": 2,
    "apacheApsVar": 2,
    "apachePatientResult": 2,
    "apachePredVar": 2    
}

for table in no_offset_tables:
    print(table)
    query = "SELECT * FROM " + table + ";"
    table_vars = pd.read_sql_query(query,conn)
    if table=="patients":
        table_vars['age'] = table_vars['age'].apply(cast_clean_int)
        num_patientunitstayids = len(table_vars["patientunitstayid"])
    table_statistics = []

    for col in tqdm(table_vars.columns[2:]):
        table_stats = pd.DataFrame({"Varname":[col]})
        if is_numeric_dtype(table_vars[col]) and (not table_vars[col].dtype == bool):
            table_stats["Val"] = ""
            quants = table_vars[col].quantile(quantiles)
            for i in range(1, len(quantiles)-1):
                table_stats["Val_"+str(quantiles[i])] = quants.iloc[i]
        else:
            u_string = ""
            unique_s = table_vars[col].unique()
            u_string = ",".join([str(x) for x in unique_s[:20]])
            if len(unique_s) >= 20:
                u_string += "..."
            table_stats["Val"] = u_string
            for i in range(1, len(quantiles)-1):
                table_stats["Val_"+str(quantiles[i])] = np.nan
        #table_stats["count"] = sum(table_vars[col].notnull())
        if "patientunitstayid" in table_vars:
            table_stats["Npata"] = len(table_vars[table_vars[col].notnull()]["patientunitstayid"].unique())
            table_stats["Npatar"] = len(table_vars[table_vars[col].notnull()]["patientunitstayid"].unique())/num_patientunitstayids
        else:
            table_stats["Npata"] = 0
            table_stats["Npatar"] = 0.
        table_statistics.append(table_stats)
    all_table_statistics = pd.concat(table_statistics, axis=0, ignore_index=True)
    all_table_statistics.to_csv(table+"_rawstatistics.csv", index=False)

### var in cols
For tables where variable names are contained in columns, extract statistics per column

In [ ]:
varcol_tables = {
    "vitalPeriodic": 3,  # large table
    #"vitalAperiodic": 3,
    #"apacheApsVar": 2,
    #"apachePatientResult": 2,
    #"apachePredVar": 2,
    #"carePlanCareProvider": 3,
    #"carePlanEOL": 2,
    #"carePlanGeneral": 4,
    #"carePlanGoal": 3,
    #"carePlanInfectiousDisease": 4,
    #"hospital": 1,
    "intakeOutput": 3, # large table
    #"note": 4,
    #"nurseAssessment": 4,
    #"respiratoryCare": 4,
    
}
for table in varcol_tables:
    print(table)
    query = "SELECT * FROM " + table + ";"
    table_vars = pd.read_sql_query(query,conn)
    has_offset = [x.lower().endswith("offset") for x in table_vars.columns[:varcol_tables[table]]]
    if any(has_offset):
        table_statistics = []
        # intermediate for stats 5-8
        relevant_rows = table_vars.filter(table_vars.columns[varcol_tables[table]:]).notna()
        var_min_max = (relevant_rows.mul(table_vars[table_vars.columns[has_offset.index(True)]], axis=0).where(relevant_rows)
                .groupby(table_vars['patientunitstayid']).agg(['min', 'max', 'count']))

        for col in tqdm(table_vars.columns[varcol_tables[table]:]):
            table_stats = pd.DataFrame({"Varname":[col]})
            if is_numeric_dtype(table_vars[col]) and (not table_vars[col].dtype == bool):
                table_stats["Val"] = ""
                quants = table_vars[col].quantile(quantiles)
                for i in range(1, len(quantiles)-1):
                    table_stats["Val_"+str(quantiles[i])] = quants.iloc[i]
            else:
                u_string = ""
                unique_s = table_vars[col].unique()
                u_string = ",".join([str(x) for x in unique_s[:20]])
                if len(unique_s) >= 20:
                    u_string += "..."
                table_stats["Val"] = u_string
                for i in range(1, len(quantiles)-1):
                    table_stats["Val_"+str(quantiles[i])] = np.nan
            #table_stats["count"] = sum(table_vars[col].notnull())
            if "patientunitstayid" in table_vars:
                table_stats["Npata"] = len(table_vars[table_vars[col].notnull()]["patientunitstayid"].unique())
                table_stats["Npatar"] = len(table_vars[table_vars[col].notnull()]["patientunitstayid"].unique())/num_patientunitstayids
            else:
                table_stats["Npata"] = 0
                table_stats["Npatar"] = 0.
            tavar_m = ((var_min_max[col]["max"] - var_min_max[col]["min"])/60).to_frame(name="Tavar")
            m = tavar_m.join(patient_demog[["patientunitstayid", "unitdischargeoffset"]].set_index("patientunitstayid"), on="patientunitstayid")
            m["unitdischargeoffset"] = m["unitdischargeoffset"]/60
            m["Trvar"] = ((var_min_max[col]["max"] - var_min_max[col]["min"])/60)/m["unitdischargeoffset"]
            m["Tvarend"] = (m["unitdischargeoffset"] - var_min_max[col]["max"]/60)/m["unitdischargeoffset"]
            m["Nppat"] = var_min_max[col]["count"]
            tvars = ["Tavar", "Trvar", "Tvarend", "Nppat"]
            for tvar in tvars:
                tavar_quants = m[m[tvar].ne(0)][tvar].quantile(quantiles_2)
                for i in range(1, len(quantiles_2)-1):
                    table_stats[tvar+str(quantiles_2[i])] = tavar_quants.iloc[i]
            table_statistics.append(table_stats)
        all_table_statistics = pd.concat(table_statistics, axis=0, ignore_index=True)
        all_table_statistics.to_csv(table+"_rawstatistics.csv", index=False)
    else:
        print("Skipping table because no offsets")

### var in rows
For tables where variable names are within rows of a single column:
- Get all distinct variable names from named column
- Get statistics per distinct variable name for all other columns

In [ ]:
# table: variable column name
varrow_tables = {
    "admissiondrug": "drugname",
    "admissiondx": "admitdxpath",
    #"allergy": "allergyname",
    #"customLab": "labothername",
    #"diagnosis": "diagnosisstring",
    #"infusionDrug": "drugname",
    #"medication": "drugname",
    #"microLab": "organism",
    #"lab": "labname",
    #"nurseCare": "celllabel",
    #"nurseCharting": "nursingchartcelltypevallabel", # largest table, took 18h
    #"pastHistory": "pasthistoryvalue",
    #"physicalExam": "physicalexamvalue",
    #"respiratoryCharting": "respchartvaluelabel",
    #"treatment": "treatmentstring"
}
for table in varrow_tables:
    print(table)
    query = "SELECT DISTINCT " + varrow_tables[table] + " FROM " + table + ";"
    distinctVars = pd.read_sql_query(query,conn)
    #print(distinctVars)
    rowtable_statistics = []
    query = "SELECT * FROM " + table + ";"
    table_vars = pd.read_sql_query(query,conn)
    has_offset = [x.lower().endswith("offset") for x in table_vars.columns]
    print("Loaded table..")
    if any(has_offset):
        relevant_columns = [x for x in table_vars.columns if (not x.endswith(("id", "offset"))) and x != varrow_tables[table]]
        for var in tqdm(distinctVars[varrow_tables[table]]):
            #search_var = var = re.sub('\'', '\'\'', str(var))    # some values contain ' , need to escape
            varvalues = table_vars.loc[table_vars[varrow_tables[table]]==var]
            table_stats = pd.DataFrame({"Varname":[var]})
            
            # intermediate for stats 5-8
            relevant_rows = varvalues.filter(relevant_columns).notna()
            var_min_max = (relevant_rows.mul(varvalues[varvalues.columns[has_offset.index(True)]], axis=0).where(relevant_rows)
                    .groupby(varvalues['patientunitstayid']).agg(['min', 'max', 'count']))

            for col in relevant_columns:
                if is_numeric_dtype(varvalues[col]) and (not varvalues[col].dtype == bool):
                    table_stats["Val_"+col] = ""
                    quants = varvalues[col].quantile(quantiles)
                    for i in range(1, len(quantiles)-1):
                        table_stats["Val_"+col+"_"+str(quantiles[i])] = quants.iloc[i]
                else:
                    u_string = ""
                    unique_s = varvalues[col].unique()
                    u_string = ",".join([str(x) for x in unique_s[:20]])
                    if len(unique_s) >= 20:
                        u_string += "..."
                    table_stats["Val_"+col] = u_string
                    for i in range(1, len(quantiles)-1):
                        table_stats["Val_"+col+"_"+str(quantiles[i])] = np.nan
                if "patientunitstayid" in varvalues:
                    table_stats["Npata_"+col] = len(varvalues[varvalues[col].notnull()]["patientunitstayid"].unique())
                    table_stats["Npatar_"+col] = len(varvalues[varvalues[col].notnull()]["patientunitstayid"].unique())/num_patientunitstayids
                else:
                    table_stats["Npata_"+col] = 0
                    table_stats["Npatar_"+col] = 0.
                tavar_m = ((var_min_max[col]["max"] - var_min_max[col]["min"])/60).to_frame(name="Tavar")
                m = tavar_m.join(patient_demog[["patientunitstayid", "unitdischargeoffset"]].set_index("patientunitstayid"), on="patientunitstayid")
                m["unitdischargeoffset"] = m["unitdischargeoffset"]/60
                m["Trvar"] = ((var_min_max[col]["max"] - var_min_max[col]["min"])/60)/m["unitdischargeoffset"]
                m["Tvarend"] = (m["unitdischargeoffset"] - var_min_max[col]["max"]/60)/m["unitdischargeoffset"]
                m["Nppat"] = var_min_max[col]["count"]
                tvars = ["Tavar", "Trvar", "Tvarend", "Nppat"]
                for tvar in tvars:
                    tavar_quants = m[m[tvar].ne(0)][tvar].quantile(quantiles_2)
                    for i in range(1, len(quantiles_2)-1):
                        table_stats[tvar+"_"+col+"_"+str(quantiles_2[i])] = tavar_quants.iloc[i]
            rowtable_statistics.append(table_stats)
        all_rowtable_statistics = pd.concat(rowtable_statistics, axis=0, ignore_index=True)
        all_rowtable_statistics.to_csv(table+"_rawstatistics.csv", index=False)
    else:
        print("Skipping table because no offsets")